# Pipeline step 4 (post_train_2): circuit-restricted fix, no quantization

Second run of this pipeline, with the fixes decided after reviewing post_train_1's
results:
- No quantization anywhere, for either arm -- plain bfloat16 throughout. The 4-bit
  QLoRA path used in post_train_1 was adopted mid-debugging as a safety margin against
  a real OOM, but the actual OOM cause was a SafeTune library bug (see safetune_bug.md
  at the repo root): PlainSFTTrainer's `_fp32_master_weights` step silently reloaded and
  upcast an orphaned, fully-unfrozen second copy of the whole base model. Since we
  bypass PlainSFTTrainer entirely with a manual PyTorch training loop (unchanged from
  post_train_1), that bug path is never hit, and our own bf16 memory estimate (~8-9GB)
  comfortably fits a T4's ~14-16GB budget without quantizing anything. This also removes
  the quantize-then-merge rounding-error risk the first run's "fixed" checkpoint carried
  (peft's own merge warning), so baseline and fixed are now loaded through the exact
  same precision path -- they differ only by whether the LoRA weights were applied.
- Trains on pipeline 3's Qwen3Guard-verified pairs -- refusals for harmful `tune`-split
  prompts plus helpful answers for their matched harmless twins (so the model also learns
  when *not* to refuse)
  (post_train_2/notebooks/combined/pipeline3_safety_pairs_v2.jsonl) instead of the original 22 pairs.
- Saves each arm's LoRA adapter separately (a few MB) before merging, so the result
  survives a Colab session timeout; verifies after training that the adapters actually changed.

Runs on a Colab G4 or T4 GPU (same session as pipeline 2, to reuse the already-downloaded
Tiny Aya weights; the kernel is restarted between notebooks so pipeline 2's model copy is not
still on the GPU). Training trims each batch's padding, so a step only pays for its real tokens.


## Start from an empty GPU

The Colab chain runs pipelines 2, 4 and 5 on one session. Without a kernel restart they share
one Python process, and anything an earlier notebook left loaded stays on the GPU. In the failed
T4 run, pipeline 2's `model` was still referenced by its `extractor`, so pipeline 4 held two bf16
copies of Tiny Aya (13.4GB of 14.5GB) before training began. The chain script now restarts the
kernel before each notebook; this cell is the backstop when a notebook is run by hand.


In [ ]:
# Drop every variable (and the Out[] history) left by earlier notebooks in this kernel.
get_ipython().run_line_magic("reset", "-f")
import gc
import sys

import torch

# A stored exception traceback keeps its frames' tensors alive after a failed cell.
for _attr in ("last_type", "last_value", "last_traceback", "last_exc"):
    if hasattr(sys, _attr):
        setattr(sys, _attr, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    _leftover_gb = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory allocated before this notebook loads anything: {_leftover_gb:.2f} GB")
    assert _leftover_gb < 1.0, (
        f"{_leftover_gb:.2f} GB is still allocated by an earlier notebook in this kernel; "
        "restart the kernel (colab restart-kernel -s SESSION) and rerun.")


> **Migrated from the `ez-penta` research repo (experiment `post_train_2`).** The executed original with its outputs from the 2026-11-12 run is [`executed/pipeline4_circuit_restricted_fix.ipynb`](executed/pipeline4_circuit_restricted_fix.ipynb). This copy only changes where files are read and written (see `configs/experiment.json`).

In [ ]:
# Project setup (Colab or local): fetch this repository's code and small data files, then load
# the shared config. Large files come from the project's Hugging Face datasets on first use.
import subprocess
import sys
from pathlib import Path

if Path("/content").exists():                     # Google Colab
    REPO = Path("/content/tiny-aya-safety")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/ishane0620/tiny-aya-safety", str(REPO)], check=True)
else:                                             # local checkout: this notebook is in <repo>/notebooks
    REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from tiny_aya_safety.config import CONFIG, adapter_source, artifact, data_path, output_dir

OUT = output_dir()   # everything this notebook writes goes here (override with TAS_OUTPUT_DIR)
print("repo:", REPO, "| outputs:", OUT)

## 0. Setup — reuse the pipeline2 session's already-downloaded model

In [ ]:
import json
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetune", "transformers", "accelerate",
                "-U", "torchao", "peft"], check=True)

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
import safetune
import safetune.interventions.harden as harden
from safetune.runner.utils.data_utils import _tokenize_qa_rows

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")

MODEL_ID = "CohereLabs/tiny-aya-water"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

# No quantization: plain bfloat16 base model. Tiny Aya Water is 3.35B parameters
# (~6.7GB in bf16); LoRA only trains a small adapter on top of this frozen base, and our
# own manual training loop never calls SafeTune's PlainSFTTrainer (see safetune_bug.md),
# so the real OOM cause from post_train_1 (a buggy fp32-master-weights upcast of an
# orphaned, fully-unfrozen second model copy) is structurally impossible here.


## 1. Load pipeline 2's target layer and pipeline 3's training pairs

In [ ]:

with open(artifact("pipeline2_results.json"), encoding="utf-8") as f:
    pipeline2_results = json.load(f)

# Pipeline 2 picks one bootstrap-stable layer per language (the layer most often in the
# top-5 safety-neuron layers across 30 resamples); the union is the restricted LoRA target.
target_layers = pipeline2_results["target_layers"]
for lang in ["en", "ja"]:
    print(f"  {lang}: stable layer {pipeline2_results[lang]['stable_layer']} "
          f"(top-5 in {pipeline2_results[lang]['stable_layer_top5_rate']:.0%} of resamples)")
print(f"Target layer(s) for the circuit-restricted fix: {target_layers}")

with open(artifact("pipeline3_safety_pairs_v2.jsonl"), encoding="utf-8") as f:
    pairs = [json.loads(line) for line in f if line.strip()]
from collections import Counter
print(f"{len(pairs)} training pairs loaded: {Counter(p.get('type', 'refusal') for p in pairs)}")
pairs[0]


## 2. Load the model fresh (or reuse, if this cell already ran earlier in this session)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
model.gradient_checkpointing_enable()
model.config.use_cache = False
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters (bf16, no quantization)")
_loaded_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory allocated after loading: {_loaded_gb:.2f} GB")
# One bf16 copy is ~6.7GB; anything near 13GB means a second copy is still loaded.
assert _loaded_gb < 9, f"{_loaded_gb:.2f} GB allocated after one model load -- a duplicate copy is resident"
print([name for name, _ in model.named_modules() if "down_proj" in name][:3], "... (example down_proj module names)")

# Sanity check: every base-model param is bf16 (no quantization anywhere). Base params are
# still requires_grad=True here -- lora_wrap() freezes them per arm below, and each arm
# prints its trainable-parameter count after wrapping.
n_non_bf16 = sum(1 for p in model.parameters() if p.dtype != torch.bfloat16)
assert n_non_bf16 == 0, f"{n_non_bf16} non-bf16 params"
print(f"\nBase-model sanity check: all params bf16 ({n_non_bf16} non-bf16)")

## 3. Build the tokenized training set

(instruction, chosen) pairs, loss on the chosen response only — reusing SafeTune's own
`_tokenize_qa_rows` so this matches exactly what its trainers expect.

In [ ]:
from datasets import Dataset

rows = [(p["instruction"], p["chosen"]) for p in pairs if p.get("chosen")]
print(f"{len(rows)} (instruction, chosen) rows")
# _tokenize_qa_rows returns a plain list of dicts, not a Dataset (that wrapping only
# happens inside build_sft_dataset, a different helper) — wrap it so .column_names /
# .with_format work in the training loop below.
train_dataset = Dataset.from_list(_tokenize_qa_rows(tokenizer, rows, max_len=768))
print(train_dataset)


## 4. Arm A — circuit-restricted LoRA: attach only to the target layer(s)' `down_proj`

`safetune.configure(lora_target_modules=...)` is the real, global knob `lora_wrap()`
reads — matches `SafetyNeuronConfig`'s default `target_module="mlp.down_proj"` from
pipeline 2, restricted to the specific layer(s) pipeline 2 identified.

**Why a manual loop, not `harden.PlainSFTTrainer`:** a direct probe proved
`lora_wrap(model)` itself is correct and cheap (417,792 fp32 params, 0.0125% of the
model, 0.00GB memory delta) — yet `PlainSFTTrainer.train()` still hit CUDA OOM at an
*identical* ~13.8GB figure across 9 different configurations (device_map, precision,
gradient checkpointing, batch size, LoRA scope). 13.8GB is suspiciously close to this
3.35B-parameter model's size in fp32 (3,349,227,520 x 4 bytes = 13.4GB) — strong
circumstantial evidence that `transformers.Trainer`'s own setup (wrapped inside
SafeTune's trainer) is dequantizing the 4-bit base model somewhere, independent of
anything configured here. Rather than keep fighting an opaque library internal, this
trains the already-validated LoRA-wrapped model directly with a plain PyTorch loop.


In [ ]:
from safetune.runner.utils.model_utils import lora_wrap, save_checkpoint
from transformers import default_data_collator
from torch.utils.data import DataLoader


def manual_lora_train(wrapped_model, dataset, epochs=1, batch_size=1, lr=1e-4):
    wrapped_model.train()
    trainable = [p for p in wrapped_model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable)
    print(f"Training {n_trainable:,} trainable parameters")
    optimizer = torch.optim.AdamW(trainable, lr=lr)
    cols = [c for c in dataset.column_names if c in ("input_ids", "attention_mask", "labels")]
    loader = DataLoader(dataset.with_format("torch", columns=cols), batch_size=batch_size, shuffle=True,
                         collate_fn=default_data_collator)
    step = 0
    for epoch in range(epochs):
        for batch in loader:
            # Every row is padded to max_len=768, and padding is labelled -100, so dropping the
            # all-padding columns leaves the loss unchanged. It cuts activation and logit memory
            # (the fp32 logits alone are ~0.8GB per 768 tokens with a 262k vocab).
            keep = batch["attention_mask"].bool().any(dim=0)
            batch = {k: v[:, keep].to(wrapped_model.device) for k, v in batch.items()}
            out = wrapped_model(**batch)
            out.loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            step += 1
            print(f"  step {step}: loss={out.loss.item():.4f}  tokens={int(keep.sum())}  "
                  f"GPU mem={torch.cuda.memory_allocated() / 1e9:.2f}GB  "
                  f"peak={torch.cuda.max_memory_allocated() / 1e9:.2f}GB")
    return wrapped_model


def assert_adapters_trained(wrapped_model):
    # LoRA's B matrices start at exactly zero, so an untouched adapter is a no-op on merge.
    lora_b = [p for n, p in wrapped_model.named_parameters() if "lora_B" in n]
    assert lora_b, "no lora_B parameters found"
    max_abs = max(p.detach().abs().max().item() for p in lora_b)
    assert max_abs > 0, "LoRA B weights are still all zero -- training did not update the adapters"
    print(f"Adapter check: {len(lora_b)} lora_B tensors, max |weight| = {max_abs:.2e} (non-zero, trained)")


restricted_target_modules = tuple(f"layers.{layer}.mlp.down_proj" for layer in target_layers)
print(f"Circuit-restricted lora_target_modules: {restricted_target_modules}")
safetune.configure(lora_target_modules=restricted_target_modules)

lora_model_restricted = lora_wrap(model)
lora_model_restricted.print_trainable_parameters()
lora_model_restricted = manual_lora_train(lora_model_restricted, train_dataset, batch_size=1, lr=1e-4)
assert_adapters_trained(lora_model_restricted)
adapter_restricted = str(OUT / "pipeline4_restricted_adapter")
lora_model_restricted.save_pretrained(adapter_restricted)
print("Circuit-restricted adapter saved:", adapter_restricted)

checkpoint_restricted = save_checkpoint(
    lora_model_restricted.merge_and_unload(), tokenizer, "pipeline4_restricted",
    out_dir=str(OUT), method="harden.circuit_restricted_lora",
    inputs=["pipeline3_safety_pairs_v2.jsonl"], params={"epochs": 1, "batch_size": 1, "lr": 1e-4},
)
print("Circuit-restricted checkpoint:", checkpoint_restricted)


## 5. Arm B — broad LoRA comparison (best-effort; same manual loop, no restriction)

Reload a fresh copy of the model and reset `lora_target_modules` back to SafeTune's
default — all `*_proj` modules, every layer — same data, same manual training loop as
Arm A, no restriction. Still wrapped in a try/except: unlike Arm A (2 layers, ~418K
params), LoRA across all ~32 layers is a genuinely larger trainable set, so there's a
real chance this still doesn't fit in 14.56GB even without the Trainer-construction
issue — that would be an honest hardware limit, not a bug, and is reported either way.


In [ ]:
import gc
import sys

try:
    del model, lora_model_restricted
except NameError:
    pass
for _attr in ("last_type", "last_value", "last_traceback", "last_exc"):
    if hasattr(sys, _attr):
        setattr(sys, _attr, None)  # a stored traceback would keep Arm A's tensors alive
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

DEFAULT_LORA_TARGET_MODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
safetune.configure(lora_target_modules=DEFAULT_LORA_TARGET_MODULES)
print(f"Broad lora_target_modules: {DEFAULT_LORA_TARGET_MODULES}")
_before_b_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory allocated before Arm B load: {_before_b_gb:.2f} GB")
if _before_b_gb > 1.0:
    print("WARNING: Arm A's model was not fully freed; Arm B may run out of memory on a T4.")

try:
    model_full = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
    model_full.gradient_checkpointing_enable()
    model_full.config.use_cache = False
    print(f"GPU memory allocated after Arm B load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    lora_model_full = lora_wrap(model_full)
    lora_model_full.print_trainable_parameters()
    lora_model_full = manual_lora_train(lora_model_full, train_dataset, batch_size=1, lr=1e-4)
    assert_adapters_trained(lora_model_full)
    adapter_full = str(OUT / "pipeline4_full_adapter")
    lora_model_full.save_pretrained(adapter_full)
    print("Broad-LoRA adapter saved:", adapter_full)

    checkpoint_full = save_checkpoint(
        lora_model_full.merge_and_unload(), tokenizer, "pipeline4_full",
        out_dir=str(OUT), method="harden.broad_lora",
        inputs=["pipeline3_safety_pairs_v2.jsonl"], params={"epochs": 1, "batch_size": 1, "lr": 1e-4},
    )
    print("Broad-LoRA checkpoint:", checkpoint_full)
    full_finetune_ok = True
except Exception as e:  # best-effort comparison arm: report and continue, don't crash the notebook
    print(f"Broad-LoRA comparison arm failed ({type(e).__name__}): {e}")
    print("Skipping this comparison arm; the circuit-restricted result above (Arm A) is the primary deliverable.")
    checkpoint_full = None
    full_finetune_ok = False


## 6. Save both checkpoints' locations for pipeline step 5 (validation)

In [ ]:
summary = {
    "target_layers": target_layers,
    "n_training_pairs": len(rows),
    "circuit_restricted_checkpoint": locals().get("checkpoint_restricted"),
    "full_finetune_checkpoint": locals().get("checkpoint_full"),
    "full_finetune_completed": locals().get("full_finetune_ok", False),
    "circuit_restricted_adapter": locals().get("adapter_restricted"),
    "full_finetune_adapter": locals().get("adapter_full"),
    "max_len": 768,
}
with open(OUT / "pipeline4_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
